In [1]:
import os
from os.path import join
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

manuscript_dir = r'C:\Users\Radovan\OneDrive\Radboud\Studentships\Jordy Thielen\Manuscript'
m_ica_dir = join(manuscript_dir, 'data', 'anova', 'ica')
m_noica_dir = join(manuscript_dir, 'data', 'anova', 'noica')

def load_decoding_results(npz_path):
    with np.load(npz_path) as data:
        epoch_bins_loaded = data['subject']
        mean_acc = data['accuracies']
        se_acc = data['ses']
    return epoch_bins_loaded, mean_acc, se_acc

epoch_bins = np.linspace(5, 80, 16, dtype=int)
task = 'covert'


# Full paths
data1_path = os.path.join(m_ica_dir, "covert_lda_alphaCSP_dec_64_ica.npz")
data2_path = os.path.join(m_noica_dir, "covert_lda_alphaCSP_dec_64_noica.npz" )
data3_path = os.path.join(m_ica_dir, "covert_lda_p300_dec_64_ica.npz")
data4_path = os.path.join(m_noica_dir, "covert_lda_p300_dec_64_noica.npz" )
data5_path = os.path.join(m_ica_dir, "covert_lda_rcca_dec_64_ica.npz" )
data6_path = os.path.join(m_noica_dir, "covert_lda_rcca_dec_64_noica.npz" )
# --...---...---...---...---...---...---...---...---...---...---...---...---...-
subj1, mean_acc1, se_acc1 = load_decoding_results(data1_path)
subj2, mean_acc2, se_acc2 = load_decoding_results(data2_path)
subj3, mean_acc3, se_acc3 = load_decoding_results(data3_path)
subj4, mean_acc4, se_acc4 = load_decoding_results(data4_path)
subj5, mean_acc5, se_acc5 = load_decoding_results(data5_path)
subj6, mean_acc6, se_acc6 = load_decoding_results(data6_path)
# --...---...---...---...---...---...---...---...---...---...---...---...---...-
data = {
    'Alpha w/ ICA': {
        'mean': mean_acc1,
        'se': se_acc1,
    },
    'Alpha wo/ ICA': {
        'mean': mean_acc2,
        'se': se_acc2,
    },
    'P300 w/ ICA': {
        'mean': mean_acc3,
        'se': se_acc3,
        
    },
    'P300 wo/ ICA': {
        'mean': mean_acc4,
        'se': se_acc4,
        

},
    'cVEP w/ ICA': {
        'mean': mean_acc5,
        'se': se_acc5,
        
    },
    'cVEP wo/ ICA': {
        'mean': mean_acc6,
        'se': se_acc6,
        

}
}

In [2]:
import pandas as pd
rows = []
n_subjects = subj1.size
for cond_label, acc in data.items():
    strategy = cond_label.split()[0]   # “P300”, “Alpha”, or “cVEP”
    ica_flag = 'With ICA' if 'w/ ICA' in cond_label else 'No ICA'
    for subj in range(n_subjects):
        rows.append({
            'subject': subj,
            'strategy': strategy,
            'ICA': ica_flag,
            'accuracy': np.mean(acc['mean'][subj])
        })
df = pd.DataFrame(rows)

In [3]:
import pandas as pd
import pingouin as pg

# pingouin expects the DV, within‐subject factors, and subject ID
aov = pg.rm_anova(
    dv='accuracy',
    within=['strategy', 'ICA'],
    subject='subject',
    data=df,
    detailed=True,       # include epsilons, W-spher, p-GG-corr
    effsize='np2'        # partial eta-squared
)

print(aov)

           Source        SS  ddof1  ddof2        MS          F         p-unc  \
0        strategy  0.717530      2     56  0.358765  25.324337  1.467276e-08   
1             ICA  0.000740      1     28  0.000740   4.173708  5.057100e-02   
2  strategy * ICA  0.003759      2     56  0.001879   5.949219  4.541412e-03   

   p-GG-corr       np2       eps  
0   0.000008  0.474911  0.580330  
1   0.050571  0.129724  1.000000  
2   0.015902  0.175239  0.594433  


In [10]:
import pingouin as pg

# ICA simple effects
for strat in df['strategy'].unique():
    sub = df[df['strategy']==strat]
    post = pg.pairwise_tests(
    dv='accuracy',
    within='strategy',    # or 'ICA' for the ICA loop
    subject='subject',
    data=df,
    parametric=True,
    padjust='bonf'        # <-- this adds a “p-corr” column
)

# now you can slice out exactly what you want:
print(post[['A','B','T','dof','p-unc','p-corr','hedges']])


       A     B          T   dof         p-unc        p-corr    hedges
0  Alpha  P300  -4.277762  28.0  1.987234e-04  5.961702e-04 -1.118313
1  Alpha  cVEP   1.629515  28.0  1.144044e-01  3.432131e-01  0.439120
2   P300  cVEP  17.377671  28.0  1.562169e-16  4.686507e-16  4.111508


In [6]:

# Strategy pairwise
post_strat = pg.pairwise_tests(dv='accuracy',
                               within='strategy',
                               subject='subject',
                               data=df,
                               padjust='bonf')
print(post_strat[['A','B','T','dof','p-unc','p-corr','hedges']])


       A     B          T   dof         p-unc        p-corr    hedges
0  Alpha  P300  -4.277762  28.0  1.987234e-04  5.961702e-04 -1.118313
1  Alpha  cVEP   1.629515  28.0  1.144044e-01  3.432131e-01  0.439120
2   P300  cVEP  17.377671  28.0  1.562169e-16  4.686507e-16  4.111508


In [11]:
# simple effects of ICA within each pipeline
for strat in ["Alpha","P300","cVEP"]:
    sub = df[df["strategy"]==strat]
    res = pg.pairwise_tests(dv="accuracy",
                            within="ICA",
                            subject="subject",
                            data=sub,
                            padjust="bonf")
    print(strat, res[['A','B','T','dof','p-unc','p-corr','hedges']])


KeyError: "['p-corr'] not in index"